# Project 2: Education Analytics
**Domain:** Education  
**Dataset Source:** `data/student_performance.csv`  

---

## Executive Overview
Statistical drivers of student performance.

---


In [ ]:
import os
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import DataLoader
from src.statistical_analysis import StatisticalAnalyzer
from src.visualization import Visualizer

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')


## 1. Data Quality & Preprocessing
Checklist:
✓ Dataset shape
✓ Missing values
✓ Duplicate rows
✓ Numeric outliers


In [ ]:
loader = DataLoader('../data/student_performance.csv')
df = loader.load_data()

print("==============================")
print("     RAW DATA QUALITY")
print("==============================")
print(loader.generate_data_quality_report())


In [ ]:
df = loader.clean_missing_values({})

print("==============================")
print("   CLEANED DATA QUALITY")
print("==============================")
print(loader.generate_data_quality_report())


### Outlier Analysis
**Business Question:** Are there unusual academic scores?


In [ ]:
sns.boxplot(data=df, x='MathScore')
plt.show()

**Finding:** A few scores fall below the lower whisker.  
**Meaning:** These are students requiring extreme intervention.  
**Recommendation:** Provide immediate tutoring for these specific IDs.

### Advanced Pandas: Groupby & Agg
**Business Question:** What is the detailed statistical breakdown of scores by Parent Education?


In [ ]:
ed_stats = df.groupby('ParentEducation').agg({'MathScore': ['mean', 'median', 'std', 'count']})
display(ed_stats)

### Statistical Hypothesis Testing (ANOVA)
**Business Question:** Is parent education associated with academic performance?


In [ ]:
stats = StatisticalAnalyzer(df)
res = stats.one_way_anova('ParentEducation', 'MathScore')
print(stats.format_hypothesis_report(
    'No difference in scores by parent education.', 'Scores differ by parent education.', 'One-Way ANOVA', 'F-stat', res['test_statistic'], res['p_value'], 'Parent education is associated with math scores.', 'No significant evidence that parent education impacts math scores in this dataset.', why_it_matters_reject='Suggests targeted interventions for at-risk demographics may be beneficial.'
))

### Statistical Hypothesis Testing (Correlation)
**Business Question:** Is higher attendance associated with better academic performance?


In [ ]:
res = stats.pearson_correlation_test('AttendancePercentage', 'MathScore')
print(stats.format_hypothesis_report(
    'No correlation between attendance and score.', 'Positive correlation exists.', 'Pearson Correlation', 'r', res['r_statistic'], res['p_value'], 'High attendance correlates with high grades.', 'No statistically significant relationship detected between attendance and math scores.', why_it_matters_reject='Supports enforcing strict attendance policies.', ci_lower=res['ci_lower'], ci_upper=res['ci_upper']
))

## Limitations
- Dataset size is limited.
- Results are observational.
- Correlation does not imply causation.
- Some variables contain missing observations.
- External factors are not included.
